# 02 — Silver | Games and Team Games (incremental)

**Project:** MLB Performance Intelligence Platform

Turns schedule payloads into two fact tables, processing only schedule files ingested since the
last successful run (watermark in `metadata.watermarks`).

| Table | Grain | Key |
|---|---|---|
| `silver.fact_game` | one row per game (completed, upcoming, postponed or suspended), incl. probable starters | `game_pk` |
| `silver.fact_team_game` | two rows per game: one per team, with runs scored/allowed and result | `game_pk, team_id` |

**Business rules**
- Scope: regular season (R) and postseason rounds (F, D, L, W).
- A game is *completed* when `codedGameState = 'F'` (includes "Completed Early"); postponed games stay in the table with `is_completed = false`, so postponements can be counted.
- A postponed or suspended game appears twice in the schedule with the same `gamePk`; the completed entry wins, and a later non-completed entry can never overwrite a completed one (guarded `MERGE`).
- `was_rescheduled` marks make-up games (the API keeps the original date in `rescheduledFrom`).
- Extra innings = innings played > scheduled innings; one-run game = final margin of 1.

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
import schemas as S
for _m in (C, P, S):
    importlib.reload(_m)
from pyspark.sql import functions as F, Window

dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "Reprocess all Bronze history")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"
RUN_ID = P.new_run_id()
CONSUMER = "silver_games"
FACT_GAME, FACT_TEAM_GAME = C.table("silver", "fact_game"), C.table("silver", "fact_team_game")

if FULL_REFRESH:  # rebuild both tables from all Bronze history (e.g. after adding a column)
    for t in (FACT_GAME, FACT_TEAM_GAME):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
wm = None if FULL_REFRESH else P.get_watermark(spark, CONSUMER)
src = P.latest_payloads(spark, "schedule", wm)
new_wm = P.max_ingested(src)
print(f"Watermark: {wm} -> {new_wm} | schedule payloads to process: {src.count()}")

In [ ]:
game_type_names = F.create_map(*[F.lit(x) for kv in C.GAME_TYPE_NAMES.items() for x in kv])

g = (src.select(F.explode(F.from_json("payload", S.SCHEDULE).dates).alias("d"), "extracted_at")
        .select(F.explode("d.games").alias("g"), "extracted_at")
        .where(F.col("g.gameType").isin(*C.GAME_TYPES)))

games = g.select(
    F.col("g.gamePk").alias("game_pk"),
    F.col("g.season").cast("int").alias("season"),
    F.col("g.gameType").alias("game_type"),
    game_type_names[F.col("g.gameType")].alias("game_type_name"),
    F.when(F.col("g.gameType") == "R", "Regular Season").otherwise("Postseason").alias("season_type"),
    F.to_date("g.officialDate").alias("official_date"),
    F.to_timestamp("g.gameDate").alias("game_datetime_utc"),
    F.col("g.status.codedGameState").alias("status_code"),
    F.col("g.status.detailedState").alias("status_detail"),
    F.col("g.status.reason").alias("status_reason"),
    F.coalesce(F.col("g.status.startTimeTBD"), F.lit(False)).alias("start_time_tbd"),
    (F.col("g.status.codedGameState") == "F").alias("is_completed"),
    F.coalesce(F.col("g.isTie"), F.lit(False)).alias("is_tie"),
    F.col("g.teams.home.team.id").alias("home_team_id"),
    F.col("g.teams.away.team.id").alias("away_team_id"),
    F.col("g.teams.home.score").alias("home_score"),
    F.col("g.teams.away.score").alias("away_score"),
    F.col("g.linescore.teams.home.hits").alias("home_hits"),
    F.col("g.linescore.teams.away.hits").alias("away_hits"),
    F.col("g.linescore.teams.home.errors").alias("home_errors"),
    F.col("g.linescore.teams.away.errors").alias("away_errors"),
    F.col("g.linescore.teams.home.leftOnBase").alias("home_left_on_base"),
    F.col("g.linescore.teams.away.leftOnBase").alias("away_left_on_base"),
    F.col("g.linescore.currentInning").alias("innings_played"),
    F.coalesce(F.col("g.linescore.scheduledInnings"), F.lit(9)).alias("scheduled_innings"),
    F.col("g.venue.id").alias("venue_id"),
    F.col("g.venue.name").alias("venue_name"),
    F.col("g.dayNight").alias("day_night"),
    F.col("g.doubleHeader").alias("double_header"),
    F.col("g.gameNumber").alias("game_number"),
    F.col("g.description").alias("game_description"),
    F.col("g.seriesDescription").alias("series_description"),
    F.col("g.seriesGameNumber").alias("series_game_number"),
    F.col("g.gamesInSeries").alias("games_in_series"),
    F.col("g.decisions.winner.id").alias("winning_pitcher_id"),
    F.col("g.decisions.loser.id").alias("losing_pitcher_id"),
    F.col("g.decisions.save.id").alias("save_pitcher_id"),
    F.col("g.teams.home.probablePitcher.id").alias("home_probable_pitcher_id"),
    F.col("g.teams.home.probablePitcher.fullName").alias("home_probable_pitcher"),
    F.col("g.teams.away.probablePitcher.id").alias("away_probable_pitcher_id"),
    F.col("g.teams.away.probablePitcher.fullName").alias("away_probable_pitcher"),
    F.to_timestamp("g.rescheduledFrom").alias("rescheduled_from_utc"),
    F.col("g.rescheduledFrom").isNotNull().alias("was_rescheduled"),
    F.col("extracted_at").alias("source_extracted_at"),
)

done = F.col("is_completed") & ~F.col("is_tie")
games = (games
    .withColumn("winning_team_id", F.when(done & (F.col("home_score") > F.col("away_score")), F.col("home_team_id"))
                                    .when(done & (F.col("away_score") > F.col("home_score")), F.col("away_team_id")))
    .withColumn("losing_team_id", F.when(done & (F.col("home_score") > F.col("away_score")), F.col("away_team_id"))
                                   .when(done & (F.col("away_score") > F.col("home_score")), F.col("home_team_id")))
    .withColumn("total_runs", F.when(F.col("is_completed"), F.col("home_score") + F.col("away_score")))
    .withColumn("run_margin", F.when(F.col("is_completed"), F.abs(F.col("home_score") - F.col("away_score"))))
    .withColumn("is_extra_innings", F.col("is_completed") & (F.col("innings_played") > F.col("scheduled_innings")))
    .withColumn("is_one_run_game", F.col("is_completed") & (F.col("run_margin") == 1))
    .withColumn("updated_at", F.current_timestamp()))

# One entry per game: completed first, then newest extraction, then latest start time
w = Window.partitionBy("game_pk").orderBy(F.col("is_completed").desc(), F.col("source_extracted_at").desc(),
                                         F.col("game_datetime_utc").desc())
games = games.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

In [ ]:
with P.log_step(spark, RUN_ID, "silver", "fact_game", str(new_wm)) as step:
    step.records = P.merge_upsert(spark, games, FACT_GAME, ["game_pk"],
        update_condition="NOT (t.is_completed AND NOT s.is_completed)",
        comment="Silver: one row per MLB game in scope (regular season + postseason), final score, line score, decisions and status")

## fact_team_game — the "team perspective" of every game
Most team KPIs (record, runs scored/allowed, home/away splits, head-to-head) are simple
aggregations over this table.

In [ ]:
def side(me, opp, is_home):
    return games.select(
        "game_pk", "season", "game_type", "season_type", "official_date", "game_datetime_utc", "is_completed",
        F.col(f"{me}_team_id").alias("team_id"),
        F.col(f"{opp}_team_id").alias("opponent_id"),
        F.lit(is_home).alias("is_home"),
        F.col(f"{me}_score").alias("runs_scored"),
        F.col(f"{opp}_score").alias("runs_allowed"),
        F.col(f"{me}_hits").alias("hits"),
        F.col(f"{opp}_hits").alias("hits_allowed"),
        F.col(f"{me}_errors").alias("errors"),
        F.col(f"{me}_left_on_base").alias("left_on_base"),
        (F.col("winning_team_id") == F.col(f"{me}_team_id")).alias("is_win"),
        (F.col("losing_team_id") == F.col(f"{me}_team_id")).alias("is_loss"),
        F.when(F.col("is_completed"), F.col(f"{me}_score") - F.col(f"{opp}_score")).alias("run_differential"),
        "venue_id", "day_night", "double_header", "game_number", "is_extra_innings", "is_one_run_game", "updated_at",
    ).withColumn("is_win", F.coalesce("is_win", F.lit(False))).withColumn("is_loss", F.coalesce("is_loss", F.lit(False)))

team_games = side("home", "away", True).unionByName(side("away", "home", False))

with P.log_step(spark, RUN_ID, "silver", "fact_team_game", str(new_wm)) as step:
    step.records = P.merge_upsert(spark, team_games, FACT_TEAM_GAME, ["game_pk", "team_id"],
        update_condition="NOT (t.is_completed AND NOT s.is_completed)",
        comment="Silver: two rows per game (one per team) with runs scored/allowed, hits, errors and result")

P.set_watermark(spark, CONSUMER, new_wm)

In [ ]:
display(spark.sql(f"""
    SELECT season, season_type, COUNT(*) AS games, SUM(INT(is_completed)) AS completed,
           SUM(INT(status_detail = 'Postponed')) AS postponed_now, SUM(INT(was_rescheduled)) AS rescheduled, ROUND(AVG(total_runs), 2) AS avg_runs_per_game
    FROM {FACT_GAME} GROUP BY season, season_type ORDER BY season, season_type"""))